# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
import getpass, os
import pandas as pd
import duckdb
from huggingface_hub import hf_hub_download


token = getpass.getpass("Enter your Hugging Face READ token: ").strip()
os.environ["HF_TOKEN"] = token

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token,
)
print("Downloaded to:", local_path)


con = duckdb.connect()

query_vector = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COALESCE(gsc_impressions, 0) as impressions,
    COALESCE(gsc_clicks, 0) as clicks,
    COALESCE(gsc_avg_position, 50.0) as avg_position,
    CASE WHEN gsc_impressions > 0 THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions ELSE 0.0 END as ctr
FROM read_parquet('{local_path}')
LIMIT 100
"""

df_features = con.execute(query_vector).df()
print("📊 Feature Vector Sample (first 100 rows):")
df_features.head()

Enter your Hugging Face READ token: ··········


fact_content_daily_performance/month=202(…):   0%|          | 0.00/124M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
📊 Feature Vector Sample (first 100 rows):


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0.000
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0.000
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0.008
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,0.000
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0.000


In [10]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_features)

https://docs.google.com/spreadsheets/d/11St1uOi53FQgHN5eNughge5xkWZiD3i5OMedVVg0-uY/edit#gid=0


In [9]:
from google.colab import auth
auth.authenticate_user()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [12]:
feature_notes = pd.DataFrame([
    {"Feature": "impressions", "Meaning": "GSC search impressions count", "Missing Value Handling": "Filled with 0 via COALESCE", "Available Before Prediction": "Yes"},
    {"Feature": "clicks", "Meaning": "GSC search clicks count", "Missing Value Handling": "Filled with 0 via COALESCE", "Available Before Prediction": "Yes"},
    {"Feature": "avg_position", "Meaning": "Average search rank position", "Missing Value Handling": "Imputed with default 50.0", "Available Before Prediction": "Yes"},
    {"Feature": "ctr", "Meaning": "Engineered Click-Through Rate", "Missing Value Handling": "Derived safely as 0.0 if impressions=0", "Available Before Prediction": "Yes"}
])

print("📋 Feature Documentation Summary")
print("=" * 100)
print(feature_notes.to_string(index=False))
print("=" * 100)


feature_notes.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('background-color', '#1e1e1e'), ('color', 'white'), ('font-weight', 'bold')]}
]).hide(axis="index")

📋 Feature Documentation Summary
     Feature                       Meaning                 Missing Value Handling Available Before Prediction
 impressions  GSC search impressions count             Filled with 0 via COALESCE                         Yes
      clicks       GSC search clicks count             Filled with 0 via COALESCE                         Yes
avg_position  Average search rank position              Imputed with default 50.0                         Yes
         ctr Engineered Click-Through Rate Derived safely as 0.0 if impressions=0                         Yes


Feature,Meaning,Missing Value Handling,Available Before Prediction
impressions,GSC search impressions count,Filled with 0 via COALESCE,Yes
clicks,GSC search clicks count,Filled with 0 via COALESCE,Yes
avg_position,Average search rank position,Imputed with default 50.0,Yes
ctr,Engineered Click-Through Rate,Derived safely as 0.0 if impressions=0,Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [17]:
import getpass, os
from huggingface_hub import hf_hub_download
import duckdb

token = getpass.getpass("Enter your Hugging Face READ token: ").strip()
os.environ["HF_TOKEN"] = token

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token,
)
print("Downloaded to:", local_path)

con = duckdb.connect()

Enter your Hugging Face READ token: ··········
Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [16]:
query_leakage_test = f"""
SELECT
    MIN(report_date) as start_window,
    MAX(report_date) as end_window,
    COUNT(*) as total_rows,
    COUNT(CASE WHEN report_date > '2026-03-31' THEN 1 END) as future_leak_count
FROM read_parquet('{local_path}')
"""

leakage_df = con.execute(query_leakage_test).df()

start = leakage_df['start_window'].iloc[0]
end = leakage_df['end_window'].iloc[0]
total = leakage_df['total_rows'].iloc[0]
leak = leakage_df['future_leak_count'].iloc[0]
leak_pct = (leak / total * 100) if total > 0 else 0

print("┌" + "─" * 58 + "┐")
print("│  🔍  DATA LEAKAGE / WINDOW INTEGRITY CHECK".ljust(59) + "│")
print("├" + "─" * 58 + "┤")
print(f"│  Date Range Start   : {str(start):<33}│")
print(f"│  Date Range End     : {str(end):<33}│")
print(f"│  Total Rows         : {total:<33,}│")
print(f"│  Future Leak Rows   : {leak:<33,}│")
print(f"│  Leak Percentage    : {leak_pct:.2f}%".ljust(59) + "│")
print("├" + "─" * 58 + "┤")

if leak == 0:
    print("│  ✅ STATUS: Clean — no future data leakage found     │")
else:
    print(f"│  ⚠️  STATUS: {leak} row(s) leak beyond window boundary".ljust(59) + "│")

print("└" + "─" * 58 + "┘")

┌──────────────────────────────────────────────────────────┐
│  🔍  DATA LEAKAGE / WINDOW INTEGRITY CHECK                │
├──────────────────────────────────────────────────────────┤
│  Date Range Start   : 2026-03-01 00:00:00              │
│  Date Range End     : 2026-03-31 00:00:00              │
│  Total Rows         : 9,841,378                        │
│  Future Leak Rows   : 0                                │
│  Leak Percentage    : 0.00%                              │
├──────────────────────────────────────────────────────────┤
│  ✅ STATUS: Clean — no future data leakage found     │
└──────────────────────────────────────────────────────────┘


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [19]:
excluded_fields = pd.DataFrame([
    {"Field": "client_hash_id", "Reason": "Kept as identifier metadata, excluded from numeric feature weights to avoid entity bias."},
    {"Field": "scroll_events", "Reason": "Post-click engagement metric that leaks interaction outcomes after the prediction point."}
])

print("🚫 Excluded Fields Summary")
print("=" * 90)
for _, row in excluded_fields.iterrows():
    print(f"• {row['Field']}")
    print(f"    → {row['Reason']}\n")
print("=" * 90)

excluded_fields.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('background-color', '#2b2b2b'), ('color', 'white'), ('font-weight', 'bold'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '8px'), ('border-bottom', '1px solid #444')]}
]).hide(axis="index")

🚫 Excluded Fields Summary
• client_hash_id
    → Kept as identifier metadata, excluded from numeric feature weights to avoid entity bias.

• scroll_events
    → Post-click engagement metric that leaks interaction outcomes after the prediction point.



Field,Reason
client_hash_id,"Kept as identifier metadata, excluded from numeric feature weights to avoid entity bias."
scroll_events,Post-click engagement metric that leaks interaction outcomes after the prediction point.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.